# Fase 6 — Deployment | CardioRisk · CRISP-DM

EWS (Early Warning Score) para detección de riesgo isquémico.

| Nivel | Umbral | Acción clínica |
|---|---|---|
| Bajo Riesgo | p < 0.30 | Monitoreo estándar. Control 24-48h |
| Riesgo Medio | 0.30 ≤ p ≤ 0.65 | Vigilancia intensiva 6-12h. ECG seriado. Biomarcadores |
| Alto Riesgo | p > 0.65 | ALERTA INMEDIATA UCI — Cardiólogo STAT |


In [ ]:
# BLOQUE 1 — INSTALACIONES + CANON ÚNICO + CONSTANTES EWS
!pip install -q kagglehub imbalanced-learn xgboost shap

import kagglehub, os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0f1a', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a2c3d',   'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',       'xtick.color': '#7a8fa8',
    'ytick.color': '#7a8fa8',      'grid.color': '#1a2c3d',
    'savefig.facecolor': '#0a0f1a'
})

# ── CANON ÚNICO (idéntico a F3/F4/F5) ──
try:
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
except:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')

df.columns = df.columns.str.lower().str.strip()
df = df.dropna()
df['slope_x_oldpeak'] = df['slope'] * df['oldpeak']
df['log_oldpeak']     = np.log1p(df['oldpeak'])
CATEGORICAL = ['gender', 'chestpain', 'restingrelectro']
TARGET = 'target'
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
EXCLUIR = [TARGET, 'patientid']
FEATURES_FINALES = [c for c in df_enc.columns if c not in EXCLUIR]
X = df_enc[FEATURES_FINALES]; y = df_enc[TARGET]
X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)
X_train,X_val,y_train,y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42,stratify=y_temp)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

# Constantes EWS — alineadas con Business Understanding (F1)
EWS_BAJO  = 0.30
EWS_MEDIO = 0.65
print(f'✓ CANON ejecutado — {len(FEATURES_FINALES)} features')
print(f'EWS: Bajo<{EWS_BAJO} | Medio {EWS_BAJO}-{EWS_MEDIO} | Alto>{EWS_MEDIO}')


In [ ]:
# BLOQUE 2 — ENTRENAR MODELO + MÉTRICAS TEST SET
model = RandomForestClassifier(max_depth=10, min_samples_split=2, n_estimators=100, random_state=42)
model.fit(X_train_sm, y_train_sm)
y_pred = model.predict(X_test_sc)
y_prob = model.predict_proba(X_test_sc)[:, 1]

recall       = recall_score(y_test, y_pred)
precision    = precision_score(y_test, y_pred)
f1           = f1_score(y_test, y_pred)
auc_roc      = roc_auc_score(y_test, y_prob)
accuracy     = accuracy_score(y_test, y_pred)
cm           = confusion_matrix(y_test, y_pred)
tn,fp,fn,tp  = cm.ravel()
error_tipo_ii = fn/(tp+fn) if (tp+fn)>0 else 0

kpi1 = '✓' if recall > 0.80 else '✗'
kpi2 = '✓' if auc_roc > 0.85 else '✗'
kpi3 = '✓' if error_tipo_ii < 0.05 else '✗'
print(f'TEST SET: Recall={recall:.4f} | AUC={auc_roc:.4f} | FN={fn}')
print(f'{kpi1} Recall>0.80 | {kpi2} AUC>0.85 | {kpi3} ErrorII<5%')


In [ ]:
# BLOQUE 3 — FUNCIÓN EWS DEPLOYABLE
def predecir_riesgo_cardiovascular(paciente: dict) -> dict:
    '''
    Predice riesgo cardiovascular EWS para un paciente.
    Args:
        paciente: dict con keys: age, gender, restingbp, serumcholestrol,
                  fastingbloodsugar, restingrelectro, maxheartrate,
                  exerciseangia, oldpeak, slope, noofmajorvessels, chestpain
    Returns:
        dict: riesgo, probabilidad, ews_nivel, accion_clinica
    '''
    df_p = pd.DataFrame([paciente])
    df_p.columns = df_p.columns.str.lower().str.strip()
    # Feature engineering — idéntico al CANON
    df_p['slope_x_oldpeak'] = df_p['slope'] * df_p['oldpeak']
    df_p['log_oldpeak']     = np.log1p(df_p['oldpeak'])
    # OHE manual — drop_first=False, mismo esquema que entrenamiento
    for cat, vals in [('gender',[0,1]),('chestpain',[0,1,2,3]),('restingrelectro',[0,1,2])]:
        v = int(df_p[cat].iloc[0])
        for val in vals:
            df_p[f'{cat}_{val}'] = 1 if v == val else 0
        df_p = df_p.drop(columns=[cat])
    # Reconstruir df limpio en orden exacto de FEATURES_FINALES — elimina columnas residuales
    row = {}
    for col in FEATURES_FINALES:
        row[col] = float(df_p[col].iloc[0]) if col in df_p.columns else 0.0
    df_final = pd.DataFrame([row])[FEATURES_FINALES]
    proba = float(model.predict_proba(scaler.transform(df_final.values))[0, 1])
    if proba < EWS_BAJO:
        nivel  = 'Bajo Riesgo'
        accion = 'Monitoreo estándar. Control en 24-48h.'
    elif proba <= EWS_MEDIO:
        nivel  = 'Riesgo Medio'
        accion = 'Vigilancia intensiva. Observación 6-12h. ECG seriado. Biomarcadores.'
    else:
        nivel  = 'Alto Riesgo'
        accion = 'ALERTA INMEDIATA UCI. Protocolo SCA. Cardiólogo STAT.'
    return {'riesgo': int(proba>=0.5), 'probabilidad': round(proba,4),
            'ews_nivel': nivel, 'accion_clinica': accion}


In [ ]:
# BLOQUE 4 — CASOS DE PRUEBA CLÍNICOS
casos = [
    ('Mujer 35a asintomática', {
        'age':35,'gender':0,'restingbp':115,'serumcholestrol':170,
        'fastingbloodsugar':0,'restingrelectro':0,'maxheartrate':158,
        'exerciseangia':0,'oldpeak':0.2,'slope':2,'noofmajorvessels':0,'chestpain':0}),
    ('Hombre 58a riesgo intermedio', {
        'age':58,'gender':1,'restingbp':138,'serumcholestrol':242,
        'fastingbloodsugar':1,'restingrelectro':1,'maxheartrate':128,
        'exerciseangia':0,'oldpeak':1.4,'slope':1,'noofmajorvessels':1,'chestpain':2}),
    ('Hombre 67a alto riesgo UCI', {
        'age':67,'gender':1,'restingbp':165,'serumcholestrol':290,
        'fastingbloodsugar':1,'restingrelectro':2,'maxheartrate':105,
        'exerciseangia':1,'oldpeak':3.8,'slope':0,'noofmajorvessels':3,'chestpain':3}),
]
print('='*65)
print('  CASOS DE PRUEBA EWS — CardioRisk')
print('='*65)
for nombre, datos in casos:
    r = predecir_riesgo_cardiovascular(datos)
    print(f'\nPaciente: {nombre}')
    print(f'  EWS:    {r["ews_nivel"]}  (p={r["probabilidad"]:.1%})')
    print(f'  Acción: {r["accion_clinica"]}')
print('='*65)
print('✓ EWS deployable — 3 casos verificados')


In [ ]:
# BLOQUE 5 — GUARDAR ARTEFACTOS
joblib.dump(model,             '/content/cardiorisk_model_v2.pkl')
joblib.dump(scaler,            '/content/cardiorisk_scaler_v2.pkl')
joblib.dump(FEATURES_FINALES,  '/content/cardiorisk_features_v2.pkl')
print('✓ Artefactos guardados: model_v2 | scaler_v2 | features_v2')


In [ ]:
# BLOQUE 6 — REPORTE CRISP-DM FINAL
s1='═'*65; s2='─'*65
print(f'\n{s1}')
print(f'{'REPORTE FINAL CRISP-DM — CARDIORISK':^65}')
print(f'{'IBM Data Science Professional Certificate':^65}')
print(f'{s1}')
print(f'\n{s2}\n1. BUSINESS UNDERSTANDING\n{s2}')
print('   Obj. Negocio : Reducir mortalidad hospitalaria 20%, optimizar UCI.')
print('   Obj. DS      : Clasificador binario con EWS auditable para riesgo isquémico.')
print('   KPIs         : Recall>80% | AUC-ROC>0.85 | Error tipo II<5%')
print(f'\n{s2}\n2. DATA UNDERSTANDING\n{s2}')
print('   Dataset      : jocelyndumlao/cardiovascular-disease-dataset (KaggleHub)')
print('   N=1000       : 14 columnas originales | patientid excluido')
print(f'\n{s2}\n3. DATA PREPARATION\n{s2}')
print(f'   Features     : {len(FEATURES_FINALES)} (9 numéricas + 2 engineered + 9 OHE)')
print('   OHE          : drop_first=False — preserva todas las categorías clínicas')
print('   Split        : 70/15/15 estratificado | Scaler fit solo en train | SMOTE solo en train')
print(f'\n{s2}\n4. MODELING\n{s2}')
print('   Ganador      : RandomForestClassifier')
print('   Params       : max_depth=10, min_samples_split=2, n_estimators=100')
print(f'\n{s2}\n5. EVALUATION — TEST SET SELLADO\n{s2}')
print(f'   Recall       : {recall:.4f}  {kpi1}')
print(f'   AUC-ROC      : {auc_roc:.4f}  {kpi2}')
print(f'   F1-Score     : {f1:.4f}')
print(f'   FN           : {fn}')
print(f'   Error tipo II: {error_tipo_ii:.4f}  {kpi3}')
print(f'\n{s2}\n6. DEPLOYMENT — EWS CardioRisk\n{s2}')
print(f'   Bajo Riesgo  : p < {EWS_BAJO}  → Monitoreo estándar')
print(f'   Riesgo Medio : {EWS_BAJO} ≤ p ≤ {EWS_MEDIO} → Vigilancia intensiva 6-12h')
print(f'   Alto Riesgo  : p > {EWS_MEDIO}  → ALERTA INMEDIATA UCI')
print(f'\n{s1}')
print(f'{'PROYECTO CARDIORISK COMPLETADO ✓':^65}')
print(f'{'20 features | RandomForest | EWS 3 niveles':^65}')
print(f'{s1}')
